In [2]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [3]:
model=load_model('model.keras')

with open('onehot_encode_geo.pkl','rb') as file:
    onehot_encode_geo=pickle.load(file)

with open('label_encoder.pkl','rb') as file:
    label_encoder=pickle.load(file)

with open('scaler.pkl','rb') as file:
    scaler=pickle.load(file)

with open('scaler_y.pkl','rb') as file:
    scaler_y=pickle.load(file)

c:\Users\shayan\Desktop\datascience_and_ai_class\salary_prediction_regression_dl\venv\Lib\site-packages\keras\src\saving\saving_lib.py:797: UserWarning: Skipping variable loading for optimizer 'rmsprop', because it has 8 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))


In [6]:
input_data={
      'CreditScore' : 0,
    'Geography': 'France',
    'Gender' : 'Male',
    'Age': 90,
    'Tenure': 1,
    'Balance': 60,
    'NumOfProducts': 1,
    'HasCrCard': 0,
    'IsActiveMember' : 1,
}

In [5]:
onehot_encode_geo.get_feature_names_out()

array(['Geography_France', 'Geography_Germany', 'Geography_Spain'],
      dtype=object)

In [ ]:
input_df = pd.DataFrame([input_data])
input_df['Gender'] = label_encoder.transform(input_df['Gender'])
geo_encoder_output = onehot_encode_geo.transform(input_df[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(
    geo_encoder_output, 
    columns=onehot_encode_geo.get_feature_names_out(['Geography'])
)
# 2. Concat and drop 'Geography' AND 'Exited' (since you dropped it in training)
# We drop Exited here so Step 5 doesn't get confused by an "extra" column
cols_to_drop = ['Geography', 'Exited']
# We use errors='ignore' just in case Exited isn't even in your input_data
input_df = pd.concat([input_df.drop(columns=cols_to_drop, errors='ignore'), geo_encoded_df], axis=1)

# 3. ALIGN COLUMNS (The Golden Rule)
# This uses the scaler's memory to pick ONLY the columns used during training
input_df = input_df[scaler.feature_names_in_]

# 4. Scale the Features
input_scaled = scaler.transform(input_df)

# 5. PREDICT 
prediction_scaled = model.predict(input_scaled)

# 6. INVERSE SCALE (Turn the decimal back into Dollars)
final_salary = scaler_y.inverse_transform(prediction_scaled)

#print(f"Predicted Salary: ${final_salary[0][0]:,.2f}")


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 107ms/step
Predicted Salary: $58,829.42


In [8]:
predicted_val = final_salary[0][0]

In [11]:
print(f"The predicted estimated salary is: ${predicted_val:,.2f}")

The predicted estimated salary is: $58,829.42


In [10]:
if predicted_val > 150000:
    print("This is considered a High-Salaray profile.")
elif predicted_val > 50000:
    print("This is considered a Mid-Range-Salary profile.")
else:
    print("This is considered a Entry-Level-Salary profile.")

This is considered a Mid-Range-Salary profile.
